# ML07 · 第一个神经网络 MLP

用 PyTorch 亲手搭一个多层感知器（multilayer perceptron, MLP），并跑完一个完整的训练循环，从零创建「网络如何学习」的直觉。

## 学习目标
- 理解多层感知器的基本结构：输入层、隐藏层（hidden layer）、输出层，以及为何需要非线性激活（nonlinear activation）。
- 学会用子类化（subclassing）nn.Module（__init__ / forward）定义自己的模型，也会用 nn.Sequential 快速堆栈。
- 认识常用积木：nn.Linear、ReLU，理解线性层与激活的分工。
- 搞懂损失函数（loss function）与优化器（optimizer）各自负责什么。
- 能独立写出标准训练循环：forward -> loss -> backward -> optimizer.step -> zero_grad，并说明每一步的意义。
- 用自造数据完成一个小回归或分类任务，观察损失下降代表「正在学」。

In [ ]:
# 概念：加载本书共用工具，并固定乱数种子让结果可重现
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)              # 固定 PyTorch 乱数，权重初始化每次相同，方便对照
np.random.seed(0)                 # 固定 numpy 乱数，自造数据每次相同

# 技巧：matplotlib 预设字体不含中文，图上一律用英文标签，避免出现方块乱码
plt.rcParams["axes.unicode_minus"] = False

print("torch 版本:", torch.__version__)
print("numpy 版本:", np.__version__)

## 从感知器到多层感知器

感知器（perceptron）是最简单的神经元：把输入加权求和再判断，本质上只能画出一条直线当分界。多层感知器（MLP）在中间多叠了隐藏层与非线性激活，才能逼近弯曲的关系。

关键直觉：线性层接线性层，数学上仍等于一个线性层（线性 + 线性还是线性）。唯有在中间插入非线性激活，网络才真正有表达力（expressive power），能画出弯曲的边界。

为什么先学这个：理解「为何需要隐藏层与激活」，后面所有结构选择才有依据。

In [ ]:
# 概念：自造一组无法用单一直线分开的点，对比直线边界与弯曲边界的差异
rng = np.random.default_rng(0)

# 造两个交错的区块：A 类分布在左下与右上，B 类分布在左上与右下（典型的 XOR 形状）
n_per_corner = 40
centers_a = [(-2.0, -2.0), (2.0, 2.0)]    # A 类两个角落
centers_b = [(-2.0, 2.0), (2.0, -2.0)]    # B 类两个角落

def make_blob(center, n):
    cx, cy = center
    return np.column_stack([rng.normal(cx, 0.7, n), rng.normal(cy, 0.7, n)])

points_a = np.vstack([make_blob(c, n_per_corner) for c in centers_a])
points_b = np.vstack([make_blob(c, n_per_corner) for c in centers_b])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax in axes:
    ax.scatter(points_a[:, 0], points_a[:, 1], c="tab:blue", label="class A", s=18)
    ax.scatter(points_b[:, 0], points_b[:, 1], c="tab:orange", label="class B", s=18)

# 左图：一条直线（单层感知器能做到的极限），怎么摆都分不开交错的两类
xs = np.linspace(-4, 4, 50)
axes[0].plot(xs, -xs, "k--", label="one straight line")   # 直线分界示意
axes[0].set_title("single line cannot separate")

# 右图：一条弯曲边界（MLP 能学到的），才能把交错的两类分开
theta = np.linspace(0, 2 * np.pi, 200)
axes[1].plot(2.6 * np.cos(theta), 2.6 * np.sin(theta), "k--", label="curved boundary")
axes[1].set_title("curved boundary can separate")

for ax in axes:
    ax.legend(loc="upper right", fontsize=8)
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 用 nn.Module 定义模型

PyTorch 的模型都继承自 nn.Module。它把模型拆成两个职责：
- __init__（构造器）：声明「这个模型有哪些层」，把层存成属性。
- forward（前向传播）：描述「一笔数据怎么依序流过这些层」。

理解这个分工是看懂所有 PyTorch 模型的钥匙：__init__ 准备积木，forward 规定数据流向。把层声明成属性后，PyTorch 会自动把它们的权重登记为模型参数（parameters），训练时才找得到要更新的对象。

形状：
```
class MyModel(nn.Module):
    def __init__(self): ...   # 准备有哪些层
    def forward(self, x): ... # 数据怎么流过这些层
```

In [ ]:
# 概念：子类化 nn.Module 写一个最小 MLP（输入 -> 隐藏 -> 输出），先不训练只看 forward 输出形状
class SmallMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()                       # 注意：一定要先调用父类别构造器，否则层不会被登记
        self.hidden = nn.Linear(in_dim, hidden_dim)   # 输入层到隐藏层的全连接
        self.act = nn.ReLU()                          # 非线性激活，赋予网络表达力
        self.output = nn.Linear(hidden_dim, out_dim)  # 隐藏层到输出层

    def forward(self, x):
        x = self.hidden(x)     # 数据先过第一层线性
        x = self.act(x)        # 再过 ReLU 折一下
        x = self.output(x)     # 最后过输出层
        return x

model = SmallMLP(in_dim=2, hidden_dim=8, out_dim=1)
print(model)               # 印出模型结构，看得到三个子模块

# 模型参数数量：把每个参数张量的元素个数加总
n_params = sum(p.numel() for p in model.parameters())
print("参数总数:", n_params)

# 造 5 笔 2 维假输入，跑一次 forward 看输出形状（此时权重是随机的，数值无意义）
dummy = torch.randn(5, 2)
out = model(dummy)         # 等同 model.forward(dummy)，但惯例用 model(x)
print("输入 shape:", dummy.shape, " 输出 shape:", out.shape)

## 网络积木：Linear、ReLU、Sequential

把模型拆成三种积木更好理解：
- nn.Linear：全连接层（fully connected layer），做加权组合，内含权重（weight）与偏置（bias）。
- ReLU：整流线性单元（rectified linear unit），把负值压成 0、正值保留，提供非线性「开关」。
- nn.Sequential：串行容器，把积木依声明顺序自动串起来，省去手写 forward。

关键是维度要接得起来：前一层的输出维度，必须等于后一层的输入维度。下面用 Sequential 重写上一节的同一个 MLP，确认两种写法等价。

In [ ]:
# 概念：用 nn.Sequential 把 Linear-ReLU-Linear 串成 MLP，并与子类化版本对照确认等价
seq_model = nn.Sequential(
    nn.Linear(2, 8),    # 输入 2 维 -> 隐藏 8 维
    nn.ReLU(),          # 非线性
    nn.Linear(8, 1),    # 隐藏 8 维 -> 输出 1 维（维度要与前一层的输出对齐）
)
print(seq_model)

# 两种写法的参数数量应一致（结构相同）
seq_params = sum(p.numel() for p in seq_model.parameters())
print("Sequential 参数总数:", seq_params, " 子类化版本:", n_params)

# 概念：拆开看 Linear 内部的 weight 与 bias 形状
first_linear = seq_model[0]                       # Sequential 可用索引取出某一层
print("第一层 weight shape:", first_linear.weight.shape)   # (输出维度, 输入维度)
print("第一层 bias shape:", first_linear.bias.shape)       # (输出维度,)

# 跑一次 forward，确认输出形状与子类化版本相同
print("Sequential 输出 shape:", seq_model(dummy).shape)

## 损失函数与优化器

训练需要两个角色分工：
- 损失函数（loss function）：给出「现在错多少」的分数。回归常用均方误差（mean squared error, MSE），分类常用交叉熵（cross-entropy）。
- 优化器（optimizer）：依梯度（gradient）决定「往哪个方向、走多大一步」去降低损失，也就是梯度下降（gradient descent）。学习率（learning rate）控制步伐大小。

交叉熵的直觉：它对「很有自信却答错」惩罚特别重，逼模型不要乱笃定。下面对同一笔假预测，分别算一次 MSE 与交叉熵，并创建一个 SGD 优化器看它绑定了哪些参数。

In [ ]:
# 概念：对同一笔假预测分别算 MSE 与交叉熵，并创建 SGD 观察它绑定的参数
import torch.optim as optim

# 回归情境：自造 4 笔预测值与真实值（例如预测楼高）
pred_reg = torch.tensor([10.0, 22.0, 8.0, 31.0])
true_reg = torch.tensor([12.0, 20.0, 9.0, 30.0])
mse_loss = nn.MSELoss()
print("MSE:", mse_loss(pred_reg, true_reg).item())   # 平均每笔误差的平方

# 分类情境：自造 3 笔、各 2 类的原始分数 logits，与正确类别索引
logits = torch.tensor([[2.0, 0.5],     # 第 1 笔比较倾向类别 0
                       [0.1, 1.8],     # 第 2 笔比较倾向类别 1
                       [1.0, 1.0]])    # 第 3 笔两类分数相同（不确定）
targets = torch.tensor([0, 1, 0])      # 正确答案
ce_loss = nn.CrossEntropyLoss()        # 注意：输入是未经 softmax 的原始 logits，不要自己先做 softmax
print("CrossEntropy:", ce_loss(logits, targets).item())

# 概念：优化器要被告知「该更新哪些参数」与「学习率」
optimizer = optim.SGD(seq_model.parameters(), lr=0.1)   # 把模型所有参数交给 SGD 管理
print("优化器管理的参数张量数量:", len(optimizer.param_groups[0]["params"]))
print("学习率 lr:", optimizer.param_groups[0]["lr"])

## 完整训练循环

所有 PyTorch 训练都是同一套五步骤骨架，每跑完整批数据一次叫一个训练周期（epoch）：
1. forward：把数据喂进模型得到预测。
2. loss：用损失函数算出错多少。
3. backward：反向传播（backpropagation）算出每个参数的梯度。
4. optimizer.step：依梯度更新参数，往损失低处走一步。
5. zero_grad：把梯度归零，准备下一轮。

特别注意：梯度预设会累加（accumulate）。若忘了 zero_grad，这一轮的梯度会叠到上一轮上，更新就会乱掉。下面对自造数据跑数十个 epoch，记录并画出损失下降曲线。

In [ ]:
# 概念：用五步骤骨架训练一个回归 MLP，记录每个 epoch 的损失并画出下降曲线
# 自造数据：楼层数对总楼高，大致成正比再加杂讯（单一特征的回归）
floors = np.linspace(3, 30, 60).reshape(-1, 1)              # 60 栋，楼层 3 到 30
heights = 3.2 * floors + 2.0 + np.random.normal(0, 2.0, floors.shape)   # 每层约 3.2 公尺加底座与杂讯

X = torch.tensor(floors, dtype=torch.float32)              # 框架惯用 float32
y = torch.tensor(heights, dtype=torch.float32)

reg_model = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))
loss_fn = nn.MSELoss()
optimizer = optim.SGD(reg_model.parameters(), lr=0.001)    # 注意：lr 太大会发散，这里用较小步伐

loss_history = []
for epoch in range(200):
    pred = reg_model(X)              # 1. forward
    loss = loss_fn(pred, y)         # 2. loss
    loss.backward()                 # 3. backward：算梯度
    optimizer.step()                # 4. step：更新参数
    optimizer.zero_grad()           # 5. zero_grad：清掉梯度，否则下一轮会累加
    loss_history.append(loss.item())

print("第一个 epoch 损失:", round(loss_history[0], 3))
print("最后一个 epoch 损失:", round(loss_history[-1], 3))

plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("loss going down means the model is learning")
plt.show()

## 实战：自造数据的小任务

现在把前面所有积木组装成一个能跑通的小项目：一个二分类（binary classification）任务。重点不只是看损失数字下降，更要学会用「画出预测结果」来判断模型好不好。

几个实战概念：
- 训练（train）与评估（evaluation）的区分：用训练数据学、再回头看模型在数据上画出的决策边界（decision boundary）。
- 过拟合（overfitting）初步直觉：若模型把杂讯也死记下来、边界扭曲得太离谱，泛化就会变差。

下面用 numpy 造两群略有重叠的点，训练一个分类 MLP，最后把决策边界画出来肉眼检查。

In [ ]:
# 概念：组装完整二分类项目，训练后画出决策边界并计算正确率
rng = np.random.default_rng(1)

# 自造两群带重叠的点：群 0 偏左下、群 1 偏右上
n = 120
g0 = rng.normal(loc=[-1.0, -1.0], scale=1.1, size=(n, 2))
g1 = rng.normal(loc=[1.5, 1.5], scale=1.1, size=(n, 2))
features_np = np.vstack([g0, g1])
labels_np = np.array([0] * n + [1] * n)

X = torch.tensor(features_np, dtype=torch.float32)
y = torch.tensor(labels_np, dtype=torch.long)            # 注意：交叉熵的标签要用整数类别 long

clf = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))   # 输出 2 类
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(clf.parameters(), lr=0.1)

for epoch in range(300):
    logits = clf(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

# 评估：取每笔分数最大的类别当预测，与真实标签比对算正确率
with torch.no_grad():                                    # 技巧：纯推论时关掉梯度计算，省内存又更快
    pred_class = clf(X).argmax(dim=1)
    accuracy = (pred_class == y).float().mean().item()
print("分类正确率:", round(accuracy, 3))

# 概念：在特征平面铺一张网格，对每个格点预测类别，画出决策边界
xx, yy = np.meshgrid(np.linspace(-5, 6, 200), np.linspace(-5, 6, 200))
grid = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32)
with torch.no_grad():
    zz = clf(grid).argmax(dim=1).numpy().reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")    # 背景色块就是决策边界划分的两区
plt.scatter(g0[:, 0], g0[:, 1], c="tab:blue", s=14, label="class 0")
plt.scatter(g1[:, 0], g1[:, 1], c="tab:red", s=14, label="class 1")
plt.legend(loc="upper left", fontsize=8)
plt.title("decision boundary, acc = %.2f" % accuracy)
plt.show()

## 练习

以下三题由浅入深，皆为复合型但简短。数据请自己用 numpy 造（仿真即可），完成后对照「验收」确认输出。

In [ ]:
# TODO 1 ·（简单）预测楼层高度（集成：nn.Module 定义 + 训练循环）
#   情境：依「楼层数」预测一栋住宅大楼的「总楼高（公尺）」，两者大致成正比但带一点杂讯。
#   要求：
#     1. 用 numpy 自造楼层数与总楼高的数据（加入随机杂讯），转成 float32 张量。
#     2. 用 nn.Module 或 nn.Sequential 定义一个小 MLP（回归，输出一个数值），用 MSE 当损失。
#     3. 写完整五步骤训练循环（forward -> loss -> backward -> step -> zero_grad）跑若干 epoch，记录损失。
#   验收：应看到损失随 epoch 明显下降，且预测楼高与真实楼高接近成一直线。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）建物用途二分类（集成：Sequential + ReLU + 交叉熵 + 训练循环）
#   情境：用两维特征（例如基地面积与容积率）判断一栋建物属于「住宅」或「商业」，两类在特征平面上略有重叠。
#   要求：
#     1. 用 numpy 自造两群带重叠的点与对应标签（标签用整数类别，转成 long 张量）。
#     2. 用 nn.Sequential 堆 Linear-ReLU-Linear、输出两类，搭配 CrossEntropyLoss 与 optim 优化器训练。
#     3. 训练后在特征平面上画出决策边界，并用 argmax 比对真实标签算出分类正确率。
#   验收：应看到一条弯曲的决策边界把两群大致分开，正确率明显高于随机猜测（0.5）。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）容积率非线性回归与容量比较（集成：多单元概念 + 一点设计思考）
#   情境：仿真「临路宽度」与「可建容积率」之间的非线性关系（例如先升后趋缓的曲线），探讨网络容量对拟合的影响。
#   要求：
#     1. 用 numpy 自造一条弯曲的非线性关系数据并加杂讯，转成张量。
#     2. 设计两个容量不同的 MLP（例如隐藏层较窄 vs. 较宽，或有无 ReLU），各自用相同训练循环训练。
#     3. 把两者的拟合曲线与损失曲线叠在一起比较，并用一段文字讨论哪个学得较好、是否有过拟合（overfitting）迹象。
#   验收：应看到含足够隐藏单元与 ReLU 的模型能贴合弯曲关系，过于简单的模型只能近似成直线，
#         并能说出容量与拟合的取舍。
# TODO: 在这里完成


## 小结

- 单层感知器只能画一条直线；多层感知器靠隐藏层加上非线性激活才有表达力，能逼近弯曲的关系。线性接线性仍是线性，激活才是关键。
- nn.Module 把模型拆成 __init__（准备有哪些层）与 forward（数据怎么流过），这是看懂所有 PyTorch 模型的钥匙。
- 三种积木分工明确：nn.Linear 做加权组合、ReLU 提供非线性开关、nn.Sequential 把层依序串起来；维度要前后接得起来。
- 损失函数给「错多少」的分数（回归用 MSE、分类用交叉熵），优化器依梯度与学习率往低处走一步。
- 训练循环固定五步骤：forward -> loss -> backward -> optimizer.step -> zero_grad；zero_grad 不能忘，因为梯度预设会累加。
- 评估模型不只看损失数字，更要画出拟合曲线或决策边界肉眼检查，并留意过拟合的迹象。